In [1]:
import requests
import bs4
from bs4 import BeautifulSoup
import requests
import re
import time
import pandas as pd

In [2]:
HEADERS = {
  "user-agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36"
}

In [3]:
occupation_mapping = {
    "admin.": "Assistente Administrativo",
    "blue-collar": "Operário de Produção",
    "housemaid": "Empregada Doméstica",
    "management": "Gestor",
    "self-employed": "Trabalhador Independente",
    "services": "Assistente de Atendimento ao Cliente",
    "technician": "Técnico"
}



def get_salary(job_title_pt):
    url = "https://pt.talent.com/salary"
    resp = requests.get(url, params={"job": job_title_pt}, headers=HEADERS, timeout=10)
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")
    page_text = soup.get_text(separator=" ")

    title_match = re.search(r'Quanto ganha um[a]?\s+(.+?)\s+em Portugal', page_text)
    matched_job = title_match.group(1) if title_match else None

    avg_match = re.search(r'€\s*([\d,]+)\s*/\s*Ano\s+Baseado em (\d+) salários', page_text)

    return {
        "job_pt_requested": job_title_pt,
        "job_pt_returned": matched_job,
        "match_ok": matched_job == job_title_pt if matched_job else False,
        "avg_salary_eur": int(avg_match.group(1).replace(',', '')) if avg_match else None,
        "n_salaries": int(avg_match.group(2)) if avg_match else None,
    }

results = []
for job_en, job_pt in occupation_mapping.items():
    row = get_salary(job_pt)
    row["job"] = job_en
    results.append(row)
    time.sleep(2) 

salary_scrape_df = pd.DataFrame(results)
#salary_df.to_csv("job_salaries_scraped.csv", index=False)

In [4]:
salary_df = salary_scrape_df[['job', 'avg_salary_eur']].copy()

still need student, entrepreneur, unemployed and retired. For entrepreneur use self-employed salary

In [5]:
import requests
url = "https://www.ine.pt/ine/json_indicador/pindica.jsp?op=2&varcd=0014532&Dim1=S7A2024&Dim2=1&Dim3=T&lang=EN"
response=requests.get(url)


In [6]:
get_pension = response.json()

In [7]:
get_pension

[{'IndicadorCod': '0014532',
  'IndicadorDsg': 'Average value of social security pensions (Serie 2017 - €/ No.) by Place of residence (NUTS - 2024) and Type of pension; Annual - Institute of Informatics',
  'MetaInfUrl': 'https://www.ine.pt/bddXplorer/htdocs/minfo.jsp?var_cd=0014532&lingua=EN',
  'DataExtracao': '2026-07-09T11:15:09.190+01:00',
  'DataUltimoAtualizacao': '2025-11-11',
  'UltimoPref': '2024',
  'Dados': {'2024': [{'geocod': '1',
     'geodsg': 'Continente',
     'dim_3': 'T',
     'dim_3_t': 'Total',
     'sinal_conv': '*',
     'sinal_conv_desc': 'Rectified value',
     'ind_string': '7 697 *',
     'valor': '7697'}]},
  'Sucesso': {'Verdadeiro': [{'Msg': 'OK'}]}}]

In [8]:
value_pension = get_pension[0]['Dados']['2024'][0]['valor']

In [9]:
salary_df

,job,avg_salary_eur
0,admin.,12600
1,blue-collar,13000
2,housemaid,14400
3,management,16800
4,self-employed,15000
5,services,13000
6,technician,11895


In [10]:
salary_df.loc[-1] = ['retired',value_pension]
salary_df.index = salary_df.index + 1
salary_df = salary_df.sort_index()

In [11]:
import requests
url = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/earn_mw_cur?format=JSON&lang=EN&geo=pt&time=2024-S1"
response=requests.get(url)


In [12]:
get_min_wage_mthly=response.json()

In [13]:
value_student_part_time = round(get_min_wage_mthly['value']['0'] * 12 /2)
value_unknown = round(get_min_wage_mthly['value']['0'] * 12)

In [14]:
salary_df.loc[-1] = ['student',value_student_part_time]
salary_df.index = salary_df.index + 1
salary_df = salary_df.sort_index()
salary_df.loc[-1] = ['unknown',value_unknown]
salary_df.index = salary_df.index + 1
salary_df = salary_df.sort_index()
salary_df.loc[-1] = ['entrepreneur', salary_df[salary_df.job=='self-employed'].avg_salary_eur.values[0]]
salary_df.index = salary_df.index + 1
salary_df = salary_df.sort_index()

In [15]:
salary_df.loc[-1] = ['unemployed',round(590.91*12)]
salary_df.index = salary_df.index + 1
salary_df = salary_df.sort_index()
#https://onevalue.gov.pt/en/custo_onevalue/unemployment-benefit/

In [16]:
salary_df

,job,avg_salary_eur
0,unemployed,7091
1,entrepreneur,15000
2,unknown,11484
3,student,5742
4,retired,7697
5,admin.,12600
6,blue-collar,13000
7,housemaid,14400
8,management,16800
9,self-employed,15000


In [17]:
salary_df.to_csv("../data/clean/job_salaries_scraped.csv", index=False, encoding= "utf-8", sep = ";")